In [ ]:
import os
import time

CACHE_DIR = 'chembl_raw'
os.makedirs(CACHE_DIR, exist_ok=True)

def fetch_cached(tid, attempts=5):
    path = os.path.join(CACHE_DIR, f'{tid}.csv')
    if os.path.exists(path):
        df = pd.read_csv(path)
    else:
        df = None
        for a in range(attempts):
            try:
                df = fetch(tid)
                break
            except Exception as e:
                wait = 20 * (a + 1)
                print(f'{tid}: attempt {a + 1} failed ({type(e).__name__}), retry in {wait}s')
                time.sleep(wait)
        if df is None or df.empty:
            raise RuntimeError(f'{tid} failed after {attempts} attempts')
        df.to_csv(path, index=False)
    df['standard_value'] = pd.to_numeric(df['standard_value'], errors='coerce')
    return df.dropna(subset=['standard_value', 'canonical_smiles'])

def build_df(df):
    df = df.copy()
    df['label'] = df.apply(label, axis=1)
    df = df.dropna(subset=['label'])
    g = df.groupby('molecule_chembl_id').agg(
        smiles=('canonical_smiles', 'first'),
        label=('label', 'nunique'),
        y=('label', 'first'),
        n_assays=('assay_chembl_id', 'nunique'),
    )
    g = g[g['label'] == 1].drop(columns='label').reset_index()
    g['y'] = g['y'].astype(int)
    return g

datasets = {}
for name, tid in targets.items():
    d = build_df(fetch_cached(tid))
    datasets[name] = d
    n1, n0 = (d['y'] == 1).sum(), (d['y'] == 0).sum()
    print(f'{name}: {n1} actives, {n0} inactives, active fraction {n1 / len(d):.2f}')
    time.sleep(2)

Output:

COX2: 3631 actives, 1134 inactives, active fraction 0.76

5LOX: 2019 actives, 300 inactives, active fraction 0.87

AChE: 4678 actives, 1409 inactives, active fraction 0.77

P-gp: 1003 actives, 191 inactives, active fraction 0.84

tubulin: 857 actives, 227 inactives, active fraction 0.79

gyrase: 384 actives, 82 inactives, active fraction 0.82

NorA: 77 actives, 1 inactives, active fraction 0.99